# Variant consequences

One row per variant, study type and consequence category. Each lead variant is annotated
with up to three sources of consequence — the most severe VEP transcript consequence, the
consequence for the molecular trait of a molQTL, and a distal fallback for intergenic
variants — plus any regulatory region (promoter or enhancer) it overlaps. Methods
"Assignment of most severe variant consequence".

Writes `variant_consequences`.

In [ ]:
import pandas as pd
from gentropy.assets.variant_consequences import VariantConsequence
from gentropy.common.session import Session
from pyspark.sql import Column
from pyspark.sql import functions as f
from pyspark.sql import types as t

from manuscript_methods import paper
from manuscript_methods.consequence import ConsequenceCategory
from manuscript_methods.datasets import LeadVariantEffect
from manuscript_methods.ve import SingleVariantEffectMethod, VariantEffect

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

REGULATORY_REGION_SCORE_CUTOFF = 0.05

In [ ]:
lve = LeadVariantEffect.from_parquet(session=session, path=paper.derived("replicated_lead_variants"))
regulatory = session.spark.read.parquet(paper.baseline("regulatoryToGene"))
target = session.spark.read.parquet(paper.release("target"))
so_terms = pd.DataFrame(list(VariantConsequence.map_sequence_ontology().items()), columns=["term", "featureId"])
print("lead variants:", lve.df.count(), "regulatory regions:", regulatory.count())

## Regulatory regions, deduplicated and prepared for an interval join

In [ ]:
regions = (
    regulatory.filter(f.col("score") >= REGULATORY_REGION_SCORE_CUTOFF)
    .dropDuplicates(["chromosome", "start", "end", "intervalType", "geneId"])
    .select(
        f.regexp_replace("chromosome", "chr", "").alias("chromosome"),
        (f.col("start") + 1).alias("iStart"),
        (f.col("end") + 1).alias("iEnd"),
        f.struct(
            f.lit(None).cast(t.LongType()).alias("distanceFromTSS"),
            f.col("geneId").alias("geneId"),
            f.col("intervalType").alias("consequenceId"),
            f.lit("interval").alias("type"),
        ).alias("intervalConsequence"),
    )
    .groupBy("chromosome", "iStart", "iEnd")
    .agg(f.array_distinct(f.collect_list("intervalConsequence")).alias("intervalConsequences"))
)
print("unique regulatory regions:", regions.count())

## The three consequence sources per lead variant

In [ ]:
def map_so_terms(so_term_col: Column, so_terms: pd.DataFrame) -> Column:
    """Map Sequence Ontology ids to their terms without a join."""
    expr = f.when(f.lit(False), None)
    for _, row in so_terms.iterrows():
        expr = expr.when(so_term_col == f.lit(row["featureId"]), f.lit(row["term"]))
    return expr


ve = VariantEffect(f.col("variantEffect")).filter_effect_by_method(SingleVariantEffectMethod.VEP)
disease_ids = f.concat_ws(",", f.sort_array(f.array_distinct("diseaseIds")))
partition = f.concat_ws(",", f.coalesce(f.col("geneId"), disease_ids), f.col("studyType"))

most_severe = f.col("leadVariantConsequence.mostSevereConsequence.transcriptConsequence")
vep_consequence = f.struct(
    most_severe["distanceFromTSS"].alias("distanceFromTSS"),
    most_severe["targetId"].alias("geneId"),
    ConsequenceCategory.classify_so_terms(
        map_so_terms(most_severe["variantFunctionalConsequenceIds"].getItem(0), so_terms)
    ).alias("consequenceId"),
    f.lit("vep").alias("type"),
)

# Only present for molQTLs whose lead variant is within 500 kb of the trait gene.
molecular = f.col("leadVariantConsequence.mostSevereConsequenceForMolecularTrait.transcriptConsequence")
qtl_consequence = f.when(
    f.col("leadVariantConsequence.mostSevereConsequenceForMolecularTrait.type") != f.lit("unknown"),
    f.struct(
        molecular["distanceFromTSS"].alias("distanceFromTSS"),
        molecular["targetId"].alias("geneId"),
        ConsequenceCategory.classify_so_terms(
            map_so_terms(molecular["variantFunctionalConsequenceIds"].getItem(0), so_terms)
        ).alias("consequenceId"),
        f.lit("qtl").alias("type"),
    ),
)

# Fallback for variants outside 500 kb of any gene, where transcript consequences are absent.
distal_consequence = f.when(
    vep_consequence["geneId"].isNull(),
    f.struct(
        f.lit(None).cast(t.LongType()).alias("distanceFromTSS"),
        f.lit(None).cast(t.StringType()).alias("geneId"),
        ConsequenceCategory.classify_so_terms(ve.assessment).alias("consequenceId"),
        f.lit("distal").alias("type"),
    ),
)

effects = lve.df.select(
    "variantId",
    f.col("variant.start").alias("vStart"),
    f.col("variant.end").alias("vEnd"),
    f.col("variant.chromosome").alias("chromosome"),
    f.col("rescaledStatistics.absEstimatedBeta").alias("absEstimatedBeta"),
    f.col("studyStatistics.studyType").alias("studyType"),
    f.filter(f.array(vep_consequence, qtl_consequence, distal_consequence), lambda x: x.isNotNull()).alias(
        "consequences"
    ),
    partition.alias("partition"),
).cache()
print("rows:", effects.count())

## Add overlapping regulatory regions and collapse to one row per consequence

In [ ]:
joined = (
    effects.join(
        regions,
        on=(
            (effects.chromosome == regions.chromosome)
            & (effects.vStart <= regions.iEnd)
            & (regions.iStart <= effects.vEnd)
        ),
        how="left",
    )
    .withColumn(
        "consequences",
        f.array_union(f.coalesce("consequences", f.array()), f.coalesce("intervalConsequences", f.array())),
    )
    .drop("chromosome", "iStart", "iEnd", "intervalConsequences")
)

collapsed = (
    joined.groupBy("variantId", "studyType", "partition")
    .agg(
        f.max("absEstimatedBeta").alias("maxAbsEstimatedBetaPerVariant"),
        f.array_distinct(f.flatten(f.collect_list("consequences"))).alias("consequences"),
        f.first("vStart").alias("vStart"),
        f.first("vEnd").alias("vEnd"),
    )
    .select(
        "variantId",
        "studyType",
        "maxAbsEstimatedBetaPerVariant",
        "vStart",
        "vEnd",
        f.inline("consequences"),
        "partition",
    )
)

## Distance to the transcription start site for regulatory consequences

In [ ]:
tss = target.select(
    f.col("id").alias("geneId"),
    f.when(f.col("canonicalTranscript.strand") == "+", f.col("canonicalTranscript.start"))
    .otherwise(f.col("canonicalTranscript.end"))
    .alias("tss"),
    f.when(f.col("canonicalTranscript.strand") == "+", 1).otherwise(-1).alias("strandDirection"),
)

closest_bound = f.when(
    f.abs(f.col("vEnd") - f.col("tss")) < f.abs(f.col("vStart") - f.col("tss")), f.col("vEnd")
).otherwise(f.col("vStart"))

consequences = (
    collapsed.join(f.broadcast(tss), on="geneId", how="left")
    .withColumn(
        "distanceFromTss",
        f.coalesce("distanceFromTss", (f.col("tss") - closest_bound) * f.col("strandDirection")),
    )
    .select(
        "studyType",
        "geneId",
        "variantId",
        f.col("maxAbsEstimatedBetaPerVariant").alias("maxAbsEstimatedBeta"),
        "distanceFromTss",
        f.col("consequenceId").alias("consequenceCategory"),
        f.col("type").alias("consequenceSource"),
        "partition",
    )
)
consequences.write.mode("overwrite").parquet(paper.derived("variant_consequences"))

written = session.spark.read.parquet(paper.derived("variant_consequences"))
print("rows:", written.count())
print("baseline rows:", session.spark.read.parquet(paper.baseline("lead_variant_consequence_exploded")).count())